# Smartphone ML Intelligence

## Part 8 — Natural-Language Recommendations with Ollama

### Objective

Let a user describe what they want in their own words, for example:

> "I watch a lot of Netflix, want something under 35k that isn't a Samsung"

and receive recommendations from the rules developed in Part 7.

### Why the language model has a small job

A first version let the model (`gemma3:1b`) translate whole sentences on its own. It got only **4 of 16** test sentences fully right. It often copied values from the examples in its prompt, for example adding "exclude Samsung" to an unrelated request, and it missed literal details such as brands, "between", and "above".

Most of a shopping request is **literal**: prices, brands, "waterproof", "AMOLED". These can be read reliably by matching words against a vocabulary extracted from the dataset. Only paraphrases ("my phone dies before lunch", "a phone for PUBG") need a language model.

### Division of work

```text
User sentence
      ↓
Keyword extraction        → prices, brands (and exclusions), processor, display,
      ↓                     requirements, features, using vocabulary from the dataset
Language model (Ollama)   → only if no feature word was found: suggest the main feature,
      ↓                     quoting the words it relied on; the quote is checked
Validation (Pydantic)     → only known values can pass
      ↓
Rules (DuckDB + Python)   → filtering, top-15% shortlist, best value (as in Part 7)
      ↓
Recommendations + the interpretation, or a clear "no match" with the reason
```

The language model never chooses or ranks phones, and it cannot add brands, prices, or requirements.

### 1. Importing & Loading

In [50]:
from math import ceil
from pathlib import Path
import json
import re
from typing import Literal

import numpy as np
import pandas as pd
import duckdb

from pydantic import BaseModel, Field, ValidationError, model_validator

The Ollama Python client is needed in addition to the packages used so far. Install it once from a terminal:

```bash
pip install ollama
```

In [51]:
import ollama

In [52]:
DATA_PATH = Path("../data/processed/phones_with_value_scores.csv")

data = pd.read_csv(
    DATA_PATH,
    parse_dates=["launch_date"]
)

print(f"Rows: {data.shape[0]}")
print(f"Columns: {data.shape[1]}")

Rows: 155
Columns: 53


### 2. Derived Columns

Four columns are derived before any request is handled. The first two are carried over from Part 7; the last two are new.

#### 2.1 Telephoto zoom (from Part 7)

In [53]:
def extract_zoom(camera_label):

    if isinstance(camera_label, str) and camera_label.startswith("Telephoto"):

        match = re.search(
            r"(\d+(?:\.\d+)?)x",
            camera_label
        )

        if match:
            return float(match.group(1))

    return np.nan


data["telephoto_zoom"] = (
    pd.concat(
        [
            data["third"].map(extract_zoom),
            data["fourth"].map(extract_zoom)
        ],
        axis=1
    )
    .max(axis=1)
)

#### 2.2 Camera sensor information (from Part 7)

All Samsung, Apple, and Google phones are treated as having sensor information, even where `camera_sensor_name` is empty.

In [54]:
SENSOR_INFO_BRANDS = ["Samsung", "Apple", "Google"]

data["camera_sensor_known"] = (
    data["camera_sensor_name"].notna()
    | data["brand"].isin(SENSOR_INFO_BRANDS)
).astype(int)

#### 2.3 Peak brightness with missing values filled

Some phones have no recorded peak brightness (`hbm`). For the multimedia rule, a missing value is replaced by the **mean brightness of phones with the same display type** (AMOLED or IPS LCD), because AMOLED panels are typically much brighter than IPS LCD panels. The original `hbm` column is left unchanged; the filled values go into a new column, `hbm_filled`.

In [55]:
hbm_mean_by_display = (
    data
    .groupby("display_type")["hbm"]
    .mean()
)

data["hbm_filled"] = data["hbm"].fillna(
    data["display_type"].map(hbm_mean_by_display)
)

pd.DataFrame({
    "mean hbm (nits)": hbm_mean_by_display.round(0),
    "phones with hbm": data.groupby("display_type")["hbm"].count(),
    "phones filled": data.groupby("display_type")["hbm"].apply(lambda s: s.isna().sum())
})

,mean hbm (nits),phones with hbm,phones filled
display_type,,,
Amoled,1606.0,100,11
IPS LCD,903.0,43,1


#### 2.4 Brand value (target encoding)

"Brand value" is measured by **target encoding** the brand with price: each phone receives the **mean price of its brand** across the currently available phones. A brand whose phones sell at higher prices on average has a higher brand value.

All phones of one brand share the same brand value, so when a user asks for brand value, the shortlist is formed by whole brands.

In [56]:
brand_value = (
    data
    .groupby("brand")["price"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "brand_value", "count": "available_phones"})
    .sort_values("brand_value", ascending=False)
)

data["brand_value"] = data["brand"].map(brand_value["brand_value"])

brand_value.round(0)

,brand_value,available_phones
brand,,
Apple,127500.0,6
Google,102286.0,7
Oppo,68143.0,14
Vivo,62214.0,14
Samsung,55885.0,26
OnePlus,41778.0,9
IQOO,40125.0,8
Xiaomi,39368.0,19
Nothing,36750.0,4


In [57]:
con = duckdb.connect()
con.register("phones", data)

print("DuckDB table registered: phones")

DuckDB table registered: phones


### 3. Vocabulary Extracted from the Dataset

The language model is only allowed to answer with names and values that actually exist in the dataset. Every unique value of the categorical columns is extracted here and later given to the model as its vocabulary ("tokens" it may use).

Because the vocabulary is read from the data, it updates automatically when new brands or values are added to the dataset.

In [58]:
VOCABULARY_COLUMNS = [
    "brand",
    "processor_type",
    "display_type",
    "speaker",
    "frame_material",
    "back_material",
    "ip_rating"
]

VOCABULARY = {
    column: sorted(data[column].dropna().unique().tolist())
    for column in VOCABULARY_COLUMNS
}

for column, values in VOCABULARY.items():
    print(f"{column:<15} {values}")

brand           ['Apple', 'Google', 'IQOO', 'Infinix', 'Lava', 'Motorola', 'Nothing', 'OnePlus', 'Oppo', 'POCO', 'Realme', 'Samsung', 'Tecno', 'Vivo', 'Xiaomi']
processor_type  ['Apple Bionic', 'Exynos', 'Mediatek', 'Snapdragon', 'Tensor', 'Unisoc']
display_type    ['Amoled', 'IPS LCD']
speaker         ['Mono', 'Stereo']
frame_material  ['Aluminium', 'Plastic', 'Titanium']
back_material   ['Aluminium', 'Glass', 'Leather', 'Plastic']
ip_rating       [52, 54, 64, 65, 66, 67, 68, 69]


### 4. What a User Can Ask For

A request is translated into three kinds of instructions.

**Filters** — which phones are allowed at all: budget, brands to include or exclude, processor type, display type.

**Requirements** — yes/no conditions a phone must meet. Each is a fixed SQL condition:

In [59]:
REQUIREMENTS = {
    "stereo_speakers": {
        "description": "Good sound: stereo speakers",
        "sql": "speaker = 'Stereo'"
    },
    "waterproof": {
        "description": "Waterproof: IP68 or IP69 rating",
        "sql": "ip_rating IN (68, 69)"
    },
    "good_build": {
        "description": "Good build: protection glass recorded, frame not plastic, back not plastic",
        "sql": "protection IS NOT NULL AND frame_material <> 'Plastic' AND back_material <> 'Plastic'"
    },
    "wireless_charging": {
        "description": "Supports wireless charging",
        "sql": "wireless_charging"
    }
}

pd.DataFrame(REQUIREMENTS).T

,description,sql
stereo_speakers,Good sound: stereo speakers,speaker = 'Stereo'
waterproof,Waterproof: IP68 or IP69 rating,"ip_rating IN (68, 69)"
good_build,"Good build: protection glass recorded, frame n...",protection IS NOT NULL AND frame_material <> '...
wireless_charging,Supports wireless charging,wireless_charging


**Ranking feature** — the one feature used to form the top-15% shortlist, exactly as in Part 7. Columns are applied in priority order: a later column only decides between phones tied on the earlier ones.

Two features are new in this notebook:

- **multimedia** — largest screen first, then highest pixel density, then highest brightness (`hbm_filled`), and stereo speakers are required
- **brand_value** — highest brand target encoding

In [60]:
FEATURES = {
    "battery": {
        "rank_by": [("battery", "higher")],
        "description": "Battery capacity"
    },
    "performance": {
        "rank_by": [("antutu", "higher")],
        "description": "AnTuTu benchmark score"
    },
    "camera": {
        "rank_by": [
            ("useful_camera_number", "higher"),
            ("camera_sensor_known", "higher")
        ],
        "description": "Useful cameras first, then documented sensor information"
    },
    "charging": {
        "rank_by": [("charging_speed", "higher")],
        "description": "Wired charging speed"
    },
    "brightness": {
        "rank_by": [("hbm", "higher")],
        "description": "Peak display brightness"
    },
    "refresh_rate": {
        "rank_by": [("refresh_rate", "higher")],
        "description": "Display refresh rate"
    },
    "slimness": {
        "rank_by": [("thickness", "lower")],
        "description": "Body thickness"
    },
    "lightness": {
        "rank_by": [("weight", "lower")],
        "description": "Weight"
    },
    "compactness": {
        "rank_by": [("size", "lower")],
        "description": "Smallest screen size"
    },
    "zoom": {
        "rank_by": [("telephoto_zoom", "higher")],
        "description": "Telephoto zoom factor"
    },
    "multimedia": {
        "rank_by": [
            ("size", "higher"),
            ("ppi", "higher"),
            ("hbm_filled", "higher")
        ],
        "requires": ["stereo_speakers"],
        "description": "Largest screen, then sharpest, then brightest; stereo speakers required"
    },
    "brand_value": {
        "rank_by": [("brand_value", "higher")],
        "description": "Brand's mean price (target encoding)"
    }
}

pd.DataFrame({
    "ranked by (in priority order)": {
        name: ", ".join(f"{column} ({direction})" for column, direction in spec["rank_by"])
        for name, spec in FEATURES.items()
    },
    "also requires": {
        name: ", ".join(spec.get("requires", [])) or "-"
        for name, spec in FEATURES.items()
    }
})

,ranked by (in priority order),also requires
battery,battery (higher),-
performance,antutu (higher),-
camera,"useful_camera_number (higher), camera_sensor_k...",-
charging,charging_speed (higher),-
brightness,hbm (higher),-
refresh_rate,refresh_rate (higher),-
slimness,thickness (lower),-
lightness,weight (lower),-
compactness,size (lower),-
zoom,telephoto_zoom (higher),-


### 5. The Structured Request

Every sentence is turned into this structure before any phone is looked up. The allowed brands, processor types, display types, requirements, and features are built from the vocabulary and dictionaries above, so nothing outside the dataset can pass validation.

"Around 30k" is stored as `around_price = 30000`, and the code turns it into a price range; no arithmetic is left to a model.

In [61]:
Brand = Literal[tuple(VOCABULARY["brand"])]
ProcessorType = Literal[tuple(VOCABULARY["processor_type"])]
DisplayType = Literal[tuple(VOCABULARY["display_type"])]
Requirement = Literal[tuple(REQUIREMENTS)]
Feature = Literal[tuple(FEATURES)]


class PhoneRequest(BaseModel):

    max_price: int | None = None
    min_price: int | None = None
    around_price: int | None = None

    brands: list[Brand] = Field(default_factory=list)
    exclude_brands: list[Brand] = Field(default_factory=list)
    processor_types: list[ProcessorType] = Field(default_factory=list)
    display_types: list[DisplayType] = Field(default_factory=list)

    requirements: list[Requirement] = Field(default_factory=list)

    rank_feature: Feature | None = None
    other_features: list[Feature] = Field(default_factory=list)

    @model_validator(mode="after")
    def check_prices(self):

        for name in ["max_price", "min_price", "around_price"]:
            value = getattr(self, name)
            if value is not None and value <= 0:
                raise ValueError(f"{name} must be positive")

        if self.min_price and self.max_price and self.min_price > self.max_price:
            raise ValueError("min_price is greater than max_price")

        return self


PhoneRequest(max_price=40000, brands=["Samsung"], rank_feature="battery")

PhoneRequest(max_price=40000, min_price=None, around_price=None, brands=['Samsung'], exclude_brands=[], processor_types=[], display_types=[], requirements=[], rank_feature='battery', other_features=[])

### 6. The Rules (No Language Model Involved)

The rules turn a `PhoneRequest` into recommendations:

1. **Filter** — availability, budget, brands, processor and display type, and all requirements, as one parameterised DuckDB query. Brand and type values come from the validated request, so only known values can reach the query.
2. **Shortlist** — keep the top 15% for the ranking feature (ties kept), as in Part 7, but **never fewer than 5 phones** (see below). If the request has no ranking feature (for example "waterproof Samsung phones under 40k"), the shortlist is formed by value score instead.
3. **Best value** — return the shortlisted phones in order of value score, **at most 5**.

#### Budget range

A budget describes the price class the user is shopping in, not just an upper limit. Someone asking for a phone "under Rs. 60,000" is not looking for a Rs. 15,000 phone, so:

| The user says | Price range searched |
|---|---|
| under / below / up to Rs. X | 0.8 × X to X, plus phones up to 1.1 × X (tagged, see below) |
| between Rs. X and Rs. Y | X to Y |
| around Rs. X | 0.9 × X to 1.1 × X |
| above / at least Rs. X | X and more |

Prices often drop below the listed price through offers, so for "under Rs. X" requests, phones priced up to **1.1 × X** are considered as well. They compete normally in the shortlist and on value score, but any that are recommended carry the note *"Could be available at Rs. X with some discounts"*, so the user can see they are above the stated budget. For "between X and Y" the range is taken exactly as given.

#### Number of recommendations

When few phones match, 15% of them is a very small number: 15% of 5 phones rounds up to just 1. Showing a single phone when five suitable ones exist is unhelpful, so the shortlist is widened:

- The shortlist is the top 15% of the matching phones, **but at least the best 5** (ties still kept).
- If fewer than 5 phones match, all of them are shortlisted.

At most 5 phones are shown, ordered by value score. So whenever 5 or more phones match, 5 are recommended, and fewer appear only when fewer phones match at all.

| Matching phones | Top 15% | Shortlist | Recommended |
|---|---|---|---|
| 3 | 1 | 3 (all) | 3 |
| 5 | 1 | 5 (all) | 5 |
| 20 | 3 | 5 (widened) | 5 |
| 60 | 9 | 9 | 5 best value |

#### No match

Some requests are impossible, for example an iPhone under Rs. 40,000. Then nothing is recommended. Instead, the rules check which part of the request rules everything out, by repeating the search with one condition removed at a time, and report it. For brand requests, the price range of that brand's available phones is shown too.

In [62]:
TOP_FRACTION = 0.15
MAX_RESULTS = 5
BUDGET_FLOOR = 0.80       # "under 60k"  -> 48,000 to 60,000
AROUND_TOLERANCE = 0.10   # "around 30k" -> 27,000 to 33,000
DISCOUNT_STRETCH = 1.10   # "under 40k" also considers phones up to 44,000, tagged


def top_fraction_shortlist(phones, rank_by, top_fraction=TOP_FRACTION, min_size=MAX_RESULTS):

    columns = [column for column, _ in rank_by]
    ascending = [direction == "lower" for _, direction in rank_by]

    rankable = phones.dropna(subset=columns)

    if rankable.empty:
        return rankable

    ranked = rankable.sort_values(columns, ascending=ascending)

    # Phones with identical values on every ranking column share a tie group (1 = best)
    tie_group = (
        ranked[columns]
        .ne(ranked[columns].shift())
        .any(axis=1)
        .cumsum()
    )

    # Top 15%, but never fewer than min_size phones (or all of them, if fewer match)
    shortlist_size = max(
        ceil(top_fraction * len(ranked)),
        min(min_size, len(ranked))
    )
    cutoff_group = tie_group.iloc[shortlist_size - 1]

    return ranked[tie_group <= cutoff_group]


def price_range(request):

    min_price, max_price = request.min_price, request.max_price

    if request.around_price is not None:
        min_price = min_price or round(request.around_price * (1 - AROUND_TOLERANCE))
        max_price = max_price or round(request.around_price * (1 + AROUND_TOLERANCE))

    # "Under X" with no lower limit: search the price class 0.8X to X
    if max_price is not None and min_price is None:
        min_price = round(max_price * BUDGET_FLOOR)

    return min_price, max_price


def discount_ceiling(request):
    """Highest price searched. For "under X", phones up to 1.1X are also considered."""

    min_price, max_price = price_range(request)

    is_under_request = (
        request.max_price is not None
        and request.min_price is None
        and request.around_price is None
    )

    if is_under_request:
        return round(max_price * DISCOUNT_STRETCH)

    return max_price


def price_text(min_price, max_price, ceiling=None):

    if min_price and max_price:
        text = f"Rs. {min_price:,} to Rs. {max_price:,}"
    elif min_price:
        text = f"from Rs. {min_price:,}"
    elif max_price:
        text = f"up to Rs. {max_price:,}"
    else:
        text = "any price"

    if ceiling and max_price and ceiling > max_price:
        text += f"; up to Rs. {ceiling:,} with possible discounts"

    return text


def budget_text(request):
    return price_text(*price_range(request), discount_ceiling(request))

In [63]:
def all_requirements(request):

    feature = FEATURES.get(request.rank_feature) if request.rank_feature else None
    extra = feature.get("requires", []) if feature else []

    return list(dict.fromkeys(request.requirements + extra))


def find_candidates(request, skip=()):
    """Run the filter query. `skip` names conditions to leave out (used to explain empty results)."""

    min_price, max_price = price_range(request)

    conditions = ["availability = 1"]
    params = []

    if "budget" not in skip:
        ceiling = discount_ceiling(request)
        if ceiling is not None:
            conditions.append("price <= ?")
            params.append(ceiling)
        if min_price is not None:
            conditions.append("price >= ?")
            params.append(min_price)

    for name, column, values, operator in [
        ("brands", "brand", request.brands, "IN"),
        ("exclude_brands", "brand", request.exclude_brands, "NOT IN"),
        ("processor_types", "processor_type", request.processor_types, "IN"),
        ("display_types", "display_type", request.display_types, "IN"),
    ]:
        if values and name not in skip:
            placeholders = ", ".join("?" for _ in values)
            conditions.append(f"{column} {operator} ({placeholders})")
            params.extend(values)

    for requirement in all_requirements(request):
        if requirement not in skip:
            conditions.append(f"({REQUIREMENTS[requirement]['sql']})")

    query = f"SELECT * FROM phones WHERE {' AND '.join(conditions)}"

    return con.execute(query, params).df()


def explain_no_match(request):

    min_price, max_price = price_range(request)

    labels = {
        "budget": f"the budget ({budget_text(request)})",
        "brands": f"the brand choice ({', '.join(request.brands)})",
        "exclude_brands": f"the excluded brands ({', '.join(request.exclude_brands)})",
        "processor_types": f"the processor choice ({', '.join(request.processor_types)})",
        "display_types": f"the display choice ({', '.join(request.display_types)})",
    }
    labels.update({
        requirement: f"the requirement '{REQUIREMENTS[requirement]['description']}'"
        for requirement in all_requirements(request)
    })

    active = [
        name for name in labels
        if (name == "budget" and (min_price or max_price))
        or (name in ("brands", "exclude_brands", "processor_types", "display_types") and getattr(request, name))
        or name in all_requirements(request)
    ]

    lines = ["No available phone matches all of these requirements."]

    for name in active:
        matches = len(find_candidates(request, skip=(name,)))
        if matches:
            lines.append(f"- Without {labels[name]}, {matches} phones would match.")

    if request.brands:
        brand_phones = data[(data["availability"] == 1) & data["brand"].isin(request.brands)]
        if not brand_phones.empty:
            lines.append(
                f"- Available {', '.join(request.brands)} phones cost "
                f"Rs. {brand_phones['price'].min():,} to Rs. {brand_phones['price'].max():,}."
            )

    if len(lines) == 1:
        lines.append("- Several conditions together rule out every phone; try relaxing more than one.")

    return "\n".join(lines)

In [64]:
def recommend_from_request(request, verbose=True):

    candidates = find_candidates(request)

    if candidates.empty:
        if verbose:
            print(explain_no_match(request))
        return pd.DataFrame(columns=["name", "brand", "price", "value_score"])

    feature = FEATURES.get(request.rank_feature) if request.rank_feature else None

    # Step 2: top-15% shortlist for the ranking feature,
    # or for the value score when no feature was requested
    if feature:
        rank_by = feature["rank_by"]
        rank_columns = [column for column, _ in rank_by]
        rank_ascending = [direction == "lower" for _, direction in rank_by]
    else:
        rank_by = [("value_score", "higher")]
        rank_columns, rank_ascending = [], []

    shortlist = top_fraction_shortlist(candidates, rank_by)

    # Step 3: best value, at most MAX_RESULTS
    result = (
        shortlist
        .sort_values(
            ["value_score"] + rank_columns + ["price"],
            ascending=[False] + rank_ascending + [True]
        )
        .head(MAX_RESULTS)
    )

    # Phones above the asked maximum are within reach only with a discount
    asked_max = price_range(request)[1]
    result = result.assign(note=[
        f"Could be available at Rs. {asked_max:,} with some discounts"
        if asked_max is not None and price > asked_max else ""
        for price in result["price"]
    ])

    if verbose:
        ranked_on = request.rank_feature if feature else "value score"
        print(
            f"{len(candidates)} phones match the filters ({budget_text(request)}) "
            f"-> {len(shortlist)} shortlisted for {ranked_on} "
            f"-> {len(result)} recommended"
        )

    display_columns = ["name", "brand", "price"] + [
        column for column in rank_columns if column != "camera_sensor_known"
    ]

    return (
        result[display_columns + ["value_score"] + (["note"] if result["note"].ne("").any() else [])]
        .round({"brand_value": 0, "hbm_filled": 0})
        .reset_index(drop=True)
    )

#### 6.1 Checking each rule by hand

Before any sentence is read, each rule is checked with a hand-written request. This confirms that the rules work on their own.

**Compact phones under Rs. 30,000** (smallest screen size)

In [65]:
recommend_from_request(
    PhoneRequest(max_price=30000, rank_feature="compactness")
)

34 phones match the filters (Rs. 24,000 to Rs. 30,000; up to Rs. 33,000 with possible discounts) -> 7 shortlisted for compactness -> 5 recommended


,name,brand,price,size,value_score,note
0,Lava Virat Curve,Lava,24000,6.67,70.3,
1,Oppo F33,Oppo,32000,6.59,40.6,"Could be available at Rs. 30,000 with some dis..."
2,Moto G77 Power,Motorola,26000,6.72,31.6,
3,Moto G Max,Motorola,28000,6.70,23.2,
4,Realme P4x,Realme,25000,6.72,22.6,


**Best battery under Rs. 40,000** (phones up to Rs. 44,000 are considered; those above Rs. 40,000 get a discount note)

In [66]:
recommend_from_request(
    PhoneRequest(max_price=40000, rank_feature="battery")
)

31 phones match the filters (Rs. 32,000 to Rs. 40,000; up to Rs. 44,000 with possible discounts) -> 5 shortlisted for battery -> 5 recommended


,name,brand,price,battery,value_score,note
0,Realme P4 Power,Realme,33000,10001,42.6,
1,Redmi Note 17 Pro,Xiaomi,37000,9000,36.1,
2,Vivo S2 FE,Vivo,39000,10000,27.7,
3,Poco X8 Power,POCO,42000,10000,18.7,"Could be available at Rs. 40,000 with some dis..."
4,Vivo T5 Pro,Vivo,42000,9020,10.3,"Could be available at Rs. 40,000 with some dis..."


**Samsung phones under Rs. 30,000** (brand filter, value picks)

In [67]:
recommend_from_request(
    PhoneRequest(max_price=30000, brands=["Samsung"])
)

2 phones match the filters (Rs. 24,000 to Rs. 30,000; up to Rs. 33,000 with possible discounts) -> 2 shortlisted for value score -> 2 recommended


,name,brand,price,value_score
0,Galaxy F56,Samsung,28000,89.0
1,Galaxy A17,Samsung,28000,5.2


**Strongest brand value under Rs. 60,000** (brand target encoding)

In [68]:
recommend_from_request(
    PhoneRequest(max_price=60000, rank_feature="brand_value")
)

19 phones match the filters (Rs. 48,000 to Rs. 60,000; up to Rs. 66,000 with possible discounts) -> 6 shortlisted for brand_value -> 5 recommended


,name,brand,price,brand_value,value_score,note
0,Vivo X200T,Vivo,60000,62214.0,99.4,
1,Oppo Reno 16c,Oppo,53000,68143.0,72.9,
2,Oppo Reno 16,Oppo,66000,68143.0,61.3,"Could be available at Rs. 60,000 with some dis..."
3,Vivo V70,Vivo,60000,62214.0,32.9,
4,Vivo V80,Vivo,63000,62214.0,21.9,"Could be available at Rs. 60,000 with some dis..."


**Good multimedia experience under Rs. 40,000** (size → ppi → brightness, stereo required)

In [69]:
recommend_from_request(
    PhoneRequest(max_price=40000, rank_feature="multimedia")
)

25 phones match the filters (Rs. 32,000 to Rs. 40,000; up to Rs. 44,000 with possible discounts) -> 5 shortlisted for multimedia -> 5 recommended


,name,brand,price,size,ppi,hbm_filled,value_score,note
0,Vivo T5,Vivo,35000,6.83,450,2000.0,56.1,
1,Iqoo Z11,IQOO,35000,6.83,450,2000.0,54.8,
2,Vivo S2 FE,Vivo,39000,6.83,450,2000.0,27.7,
3,Vivo S2,Vivo,43000,6.89,450,1606.0,14.8,"Could be available at Rs. 40,000 with some dis..."
4,Vivo T5 Pro,Vivo,42000,6.83,450,2000.0,10.3,"Could be available at Rs. 40,000 with some dis..."


**Good sound under Rs. 20,000** (stereo speakers)

In [70]:
recommend_from_request(
    PhoneRequest(max_price=20000, requirements=["stereo_speakers"])
)

4 phones match the filters (Rs. 16,000 to Rs. 20,000; up to Rs. 22,000 with possible discounts) -> 4 shortlisted for value score -> 4 recommended


,name,brand,price,value_score,note
0,Oppo K14,Oppo,20000,87.1,
1,Moto G37 Power,Motorola,16000,83.2,
2,Poco M8x,POCO,21000,60.6,"Could be available at Rs. 20,000 with some dis..."
3,Tecno Spark 50s,Tecno,20000,41.3,


**Waterproof phones under Rs. 25,000** (IP68 or IP69)

In [71]:
recommend_from_request(
    PhoneRequest(max_price=25000, requirements=["waterproof"])
)

5 phones match the filters (Rs. 20,000 to Rs. 25,000; up to Rs. 27,500 with possible discounts) -> 5 shortlisted for value score -> 5 recommended


,name,brand,price,value_score
0,Moto Edge 60 Fusion,Motorola,23000,91.6
1,Oppo K14,Oppo,20000,87.1
2,Realme Narzo 100x,Realme,21000,83.9
3,Infinix Hot 70 Pro,Infinix,25000,74.8
4,Realme P4R,Realme,23000,67.7


**Well-built phones under Rs. 50,000** (protection recorded, no plastic frame or back)

In [72]:
recommend_from_request(
    PhoneRequest(max_price=50000, requirements=["good_build"])
)

8 phones match the filters (Rs. 40,000 to Rs. 50,000; up to Rs. 55,000 with possible discounts) -> 5 shortlisted for value score -> 5 recommended


,name,brand,price,value_score,note
0,Moto Edge 70 Max,Motorola,55000,100.0,"Could be available at Rs. 50,000 with some dis..."
1,Iqoo 15R,IQOO,54000,97.4,"Could be available at Rs. 50,000 with some dis..."
2,Poco X8 Pro Max,POCO,49000,94.8,
3,Oneplus 13s,OnePlus,55000,94.2,"Could be available at Rs. 50,000 with some dis..."
4,Poco X8 Pro,POCO,40000,73.5,


**An impossible request: an iPhone under Rs. 40,000**

In [73]:
recommend_from_request(
    PhoneRequest(max_price=40000, brands=["Apple"])
)

No available phone matches all of these requirements.
- Without the budget (Rs. 32,000 to Rs. 40,000; up to Rs. 44,000 with possible discounts), 6 phones would match.
- Without the brand choice (Apple), 31 phones would match.
- Available Apple phones cost Rs. 80,000 to Rs. 180,000.


,name,brand,price,value_score


**Another impossible request: a waterproof, well-built phone under Rs. 15,000**

In [74]:
recommend_from_request(
    PhoneRequest(max_price=15000, requirements=["waterproof", "good_build"])
)

No available phone matches all of these requirements.
- Without the budget (Rs. 12,000 to Rs. 15,000; up to Rs. 16,500 with possible discounts), 45 phones would match.


,name,brand,price,value_score


### 7. Reading a Sentence: Keyword Extraction

The first and main step of reading a sentence uses no language model. Words in the sentence are matched against:

- **aliases built from the dataset vocabulary** — every brand, processor type, and display type, plus everyday names such as "iPhone" (Apple), "Redmi" (Xiaomi), or "Dimensity" (MediaTek)
- **requirement words** — for example "waterproof", "rain", or "IP68" for the waterproof requirement
- **feature words** — for example "battery", "backup", or "lasts" for battery

Longer phrases are matched first, so "wireless charging" counts as the wireless-charging requirement rather than the charging feature, and "low light" counts as camera rather than lightness.

#### 7.1 Vocabulary aliases

In [75]:
SUB_BRANDS = {
    "iphone": "Apple",
    "pixel": "Google",
    "galaxy": "Samsung",
    "redmi": "Xiaomi",
    "narzo": "Realme",
    "nord": "OnePlus",
    "one plus": "OnePlus",
    "moto": "Motorola",
    "cmf": "Nothing",
    "nothing phone": "Nothing",
}

# "Nothing" is also an ordinary English word ("nothing too expensive"),
# so the brand is only recognised as "Nothing Phone" or "CMF"
AMBIGUOUS_BRAND_WORDS = {"nothing"}

BRAND_ALIASES = {
    brand.lower(): brand
    for brand in VOCABULARY["brand"]
    if brand.lower() not in AMBIGUOUS_BRAND_WORDS
}
BRAND_ALIASES.update(SUB_BRANDS)

PROCESSOR_ALIASES = {processor.lower(): processor for processor in VOCABULARY["processor_type"]}
PROCESSOR_ALIASES.update({"dimensity": "Mediatek", "helio": "Mediatek", "bionic": "Apple Bionic"})

DISPLAY_ALIASES = {display.lower(): display for display in VOCABULARY["display_type"]}
DISPLAY_ALIASES.update({"oled": "Amoled", "poled": "Amoled", "p-oled": "Amoled", "lcd": "IPS LCD", "ips": "IPS LCD"})

print("Brand aliases     :", BRAND_ALIASES)
print("Processor aliases :", PROCESSOR_ALIASES)
print("Display aliases   :", DISPLAY_ALIASES)

Brand aliases     : {'apple': 'Apple', 'google': 'Google', 'iqoo': 'IQOO', 'infinix': 'Infinix', 'lava': 'Lava', 'motorola': 'Motorola', 'oneplus': 'OnePlus', 'oppo': 'Oppo', 'poco': 'POCO', 'realme': 'Realme', 'samsung': 'Samsung', 'tecno': 'Tecno', 'vivo': 'Vivo', 'xiaomi': 'Xiaomi', 'iphone': 'Apple', 'pixel': 'Google', 'galaxy': 'Samsung', 'redmi': 'Xiaomi', 'narzo': 'Realme', 'nord': 'OnePlus', 'one plus': 'OnePlus', 'moto': 'Motorola', 'cmf': 'Nothing', 'nothing phone': 'Nothing'}
Processor aliases : {'apple bionic': 'Apple Bionic', 'exynos': 'Exynos', 'mediatek': 'Mediatek', 'snapdragon': 'Snapdragon', 'tensor': 'Tensor', 'unisoc': 'Unisoc', 'dimensity': 'Mediatek', 'helio': 'Mediatek', 'bionic': 'Apple Bionic'}
Display aliases   : {'amoled': 'Amoled', 'ips lcd': 'IPS LCD', 'oled': 'Amoled', 'poled': 'Amoled', 'p-oled': 'Amoled', 'lcd': 'IPS LCD', 'ips': 'IPS LCD'}


#### 7.2 Requirement and feature words

In [76]:
REQUIREMENT_TERMS = {
    "stereo_speakers": ["stereo", "speaker", "speakers", "sound", "audio", "music", "loud"],
    "waterproof": [
        "waterproof", "water proof", "water-proof", "water resistant", "water-resistant",
        "water resistance", "rain", "rainproof", "pool", "swim", "swimming", "underwater", "ip68", "ip69"
    ],
    "good_build": [
        "build", "built", "build quality", "sturdy", "solid", "durable", "rugged", "metal",
        "plastic", "plasticky", "premium build", "premium feel", "feels premium"
    ],
    "wireless_charging": ["wireless charging", "wireless"],
}

FEATURE_TERMS = {
    "battery": [
        "battery", "batteries", "battery life", "backup", "long lasting", "long-lasting",
        "lasts", "mah", "endurance", "dies"
    ],
    "performance": [
        "performance", "gaming", "games", "game", "gamer", "powerful", "processor", "chipset",
        "lag", "lag-free", "antutu", "benchmark", "speed"
    ],
    "camera": [
        "camera", "cameras", "photo", "photos", "photography", "picture", "pictures", "pics",
        "selfie", "selfies", "low light", "low-light", "night photos", "instagram",
        "video recording", "record videos", "vlogging"
    ],
    "charging": [
        "charging", "charge", "charges", "charger", "fast charging", "quick charging",
        "charging speed", "watt", "watts"
    ],
    "brightness": ["bright", "brightness", "brightest", "sunlight", "outdoor", "outdoors", "nits"],
    "refresh_rate": ["refresh rate", "refresh", "smooth scrolling", "smooth display", "120hz", "144hz", "165hz"],
    "slimness": ["slim", "slimmest", "thin", "thinnest", "sleek", "thickness"],
    "lightness": ["light", "lightweight", "light weight", "lightest", "weight", "not heavy"],
    "compactness": [
        "compact", "small", "smallest", "smaller", "one hand", "one-hand", "one handed",
        "mini", "pocket", "pocketable"
    ],
    "zoom": ["zoom", "telephoto", "periscope"],
    "multimedia": [
        "multimedia", "movie", "movies", "netflix", "youtube", "video", "videos", "binge",
        "series", "streaming", "entertainment", "big screen", "large screen", "bigger screen", "content"
    ],
    "brand_value": [
        "brand value", "premium brand", "prestigious", "prestige", "status", "brand name",
        "luxury", "reputed brand"
    ],
}

assert set(REQUIREMENT_TERMS) == set(REQUIREMENTS)
assert set(FEATURE_TERMS) == set(FEATURES)

print("Every requirement and feature has its own word list.")

Every requirement and feature has its own word list.


#### 7.3 Prices

Prices are read with regular expressions:

- amounts like `40k`, `40,000`, `Rs 40000`, `35 thousand`, `1 lakh`, `1.5 lakh`
- ranges like `between 15k and 25k`, `20 to 30k`, `15000-25000`
- "around", "about", "roughly" → approximate budget
- "above", "at least", "more than" → minimum; "under", "below", "within", "up to", "budget" → maximum
- numbers that are clearly not prices are ignored: model numbers ("Galaxy S25", "iPhone 16"), and values followed by units such as mAh, W, Hz, GB, or MP

In [77]:
UNIT = r"(k|thousand|lakhs?|lacs?|l)"
NOT_A_PRICE_AFTER = r"(?!\s*(?:mah|w\b|watts?|hz|nits|mp|gb|tb|inch|inches|mm|g\b|grams|%))"
AMOUNT = rf"(?<![a-z0-9.])(\d+(?:\.\d+)?)\s*{UNIT}?(?![a-z0-9]){NOT_A_PRICE_AFTER}"

MIN_CUES = {
    "above", "over", "more than", "at least", "atleast", "minimum", "min",
    "starting", "from", "greater than", "upwards of"
}
MAX_CUES = {
    "under", "below", "within", "up to", "upto", "max", "maximum", "less than", "budget",
    "budget of", "for", "at most", "cheaper than", "till", "until", "not more than", "lower than"
}
POST_MAX_CUES = {"max", "maximum", "or less", "or below", "or under", "budget"}
POST_MIN_CUES = {"or more", "plus", "+", "and above", "or above", "onwards"}


def normalise(sentence):

    text = sentence.lower()
    text = text.replace("₹", " rs ")
    text = re.sub(r"(?<=\d),(?=\d)", "", text)      # 45,000 -> 45000
    text = re.sub(r"\brs\.?\s*", "rs ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def to_rupees(number, unit):

    value = float(number)

    if unit in ("k", "thousand"):
        value *= 1_000
    elif unit and unit.startswith(("lakh", "lac", "l")):
        value *= 100_000

    return int(round(value))


def tokens(text):
    return re.findall(r"[a-z0-9'+]+|,", text)


def cue_before(text, start, cues, max_tokens=3):
    """Distance (in words) to the nearest cue among the last few words before an amount, or None."""

    words = [w for w in tokens(text[:start]) if w not in ("rs", "inr", "rupees")][-max_tokens:]

    for i in range(len(words) - 1, -1, -1):
        candidates = {words[i], " ".join(words[max(0, i - 1):i + 1]), " ".join(words[max(0, i - 2):i + 1])}
        if candidates & cues:
            return len(words) - i

    return None


def cue_after(text, end, cues):
    following = text[end:end + 15].strip()
    return any(following.startswith(cue) for cue in cues)


def extract_prices(text):

    prices = {"min_price": None, "max_price": None, "around_price": None}
    used = []

    # Ranges: "between 15k and 25k", "20 to 30k", "15000-25000"
    range_pattern = rf"(?:between\s+|from\s+)?{AMOUNT}\s*(?:and|to|-|–)\s*(?:rs\s*)?{AMOUNT}"

    for match in re.finditer(range_pattern, text):
        low_number, low_unit, high_number, high_unit = match.groups()
        low = to_rupees(low_number, low_unit or high_unit)    # "20 to 30k": 20 means 20k
        high = to_rupees(high_number, high_unit)
        if low >= 1000 and high >= 1000:
            prices["min_price"], prices["max_price"] = min(low, high), max(low, high)
            used.append(match.span())
            break

    # Approximate: "around 30k", "about 1 lakh"
    around_pattern = rf"\b(?:around|about|approx(?:imately)?|roughly|near|close to|~)\s*(?:rs\s*)?{AMOUNT}"

    for match in re.finditer(around_pattern, text):
        if any(s <= match.start() < e for s, e in used):
            continue
        value = to_rupees(*match.groups())
        if value >= 1000:
            prices["around_price"] = value
            used.append(match.span())
            break

    # Single amounts, with a cue before or after them
    for match in re.finditer(AMOUNT, text):

        if any(s <= match.start() < e for s, e in used):
            continue

        value = to_rupees(*match.groups())
        if value < 1000:
            continue

        min_distance = cue_before(text, match.start(), MIN_CUES)
        max_distance = cue_before(text, match.start(), MAX_CUES)

        if min_distance and (not max_distance or min_distance < max_distance):
            kind = "min_price"
        elif max_distance:
            kind = "max_price"
        elif cue_after(text, match.end(), POST_MIN_CUES):
            kind = "min_price"
        else:
            kind = "max_price"          # a bare amount is read as the budget

        if prices[kind] is None:
            prices[kind] = value
        used.append(match.span())

    return prices, used


for example in ["under 40k", "between 15k and 25k", "20 to 30k", "around 1.5 lakh",
                "above Rs 60,000", "50k max", "5000 mAh battery under 15k", "galaxy s25 for 80000"]:
    print(f"{example:<28} -> {extract_prices(normalise(example))[0]}")

under 40k                    -> {'min_price': None, 'max_price': 40000, 'around_price': None}
between 15k and 25k          -> {'min_price': 15000, 'max_price': 25000, 'around_price': None}
20 to 30k                    -> {'min_price': 20000, 'max_price': 30000, 'around_price': None}
around 1.5 lakh              -> {'min_price': None, 'max_price': None, 'around_price': 150000}
above Rs 60,000              -> {'min_price': 60000, 'max_price': None, 'around_price': None}
50k max                      -> {'min_price': None, 'max_price': 50000, 'around_price': None}
5000 mAh battery under 15k   -> {'min_price': None, 'max_price': 15000, 'around_price': None}
galaxy s25 for 80000         -> {'min_price': None, 'max_price': 80000, 'around_price': None}


#### 7.4 Matching words and handling "not"

Brands can be wanted or unwanted. A brand counts as **excluded** when a negation word comes right before it, possibly with other brands in between: "anything except **vivo and oppo**", "isn't a **Xiaomi**", "not a **samsung or apple**".

In [78]:
NEGATION_WORDS = {"except", "not", "no", "without", "excluding", "exclude", "avoid", "isn't", "isnt", "nor", "hate"}
NEGATION_PHRASES = {
    "anything but", "other than", "apart from", "rather than", "instead of",
    "don't want", "dont want", "do not want", "but not"
}
LIST_WORDS = {"and", "or", "nor", ",", "a", "an", "the", "phone", "phones", "from"}
BRAND_WORDS = {word for alias in BRAND_ALIASES for word in alias.split()} | AMBIGUOUS_BRAND_WORDS


def is_negated(text, start):

    words = tokens(text[:start])

    # Skip back over a list of brands: "except vivo and oppo"
    while words and (words[-1] in LIST_WORDS or words[-1] in BRAND_WORDS):
        words.pop()

    if not words:
        return False

    return words[-1] in NEGATION_WORDS or " ".join(words[-2:]) in NEGATION_PHRASES


TERM_TABLE = (
    [(alias, "brand", brand) for alias, brand in BRAND_ALIASES.items()]
    + [(alias, "processor", processor) for alias, processor in PROCESSOR_ALIASES.items()]
    + [(alias, "display", display) for alias, display in DISPLAY_ALIASES.items()]
    + [(term, "requirement", name) for name, terms in REQUIREMENT_TERMS.items() for term in terms]
    + [(term, "feature", name) for name, terms in FEATURE_TERMS.items() for term in terms]
)


def find_terms(text, used_spans):
    """All known terms in the text. Longest phrases first; matches never overlap; prices are skipped."""

    masked = list(text)
    for s, e in used_spans:
        masked[s:e] = "#" * (e - s)

    found = []

    for phrase, category, value in sorted(TERM_TABLE, key=lambda row: -len(row[0])):
        pattern = rf"(?<![a-z0-9]){re.escape(phrase)}(?![a-z0-9])"
        for match in re.finditer(pattern, "".join(masked)):
            found.append((match.start(), category, value, phrase))
            masked[match.start():match.end()] = "#" * (match.end() - match.start())

    return sorted(found)

#### 7.5 The keyword extractor

The extractor combines prices and terms into a `PhoneRequest`. If several features are mentioned, the first one in the sentence becomes the ranking feature and the rest are reported as "also mentioned".

In [79]:
def keyword_extract(sentence):

    text = normalise(sentence)
    prices, used = extract_prices(text)

    found = dict(brands=[], exclude_brands=[], processor_types=[], display_types=[], requirements=[])
    features_in_order = []
    matched_words = []

    for start, category, value, phrase in find_terms(text, used):

        matched_words.append(phrase)

        if category == "brand":
            target = "exclude_brands" if is_negated(text, start) else "brands"
        elif category == "processor":
            target = "processor_types"
        elif category == "display":
            target = "display_types"
        elif category == "requirement":
            target = "requirements"
        else:
            if value not in features_in_order:
                features_in_order.append(value)
            continue

        if value not in found[target]:
            found[target].append(value)

    request = PhoneRequest(
        **prices,
        **found,
        rank_feature=features_in_order[0] if features_in_order else None,
        other_features=features_in_order[1:]
    )

    return request, matched_words


for example in [
    "anything except vivo and oppo with fast charging below 30k",
    "a long lasting phone that survives rain and isn't a Xiaomi, around 35k",
    "nothing too fancy, around 20k",
    "Nothing Phone with a good camera under 40k",
    "waterproof phone with wireless charging and a good camera under 70k",
]:
    request, words = keyword_extract(example)
    print(example)
    print("   ", request.model_dump_json(exclude_defaults=True))
    print("    matched:", words, "\n")

anything except vivo and oppo with fast charging below 30k
    {"max_price":30000,"exclude_brands":["Vivo","Oppo"],"rank_feature":"charging"}
    matched: ['vivo', 'oppo', 'fast charging'] 

a long lasting phone that survives rain and isn't a Xiaomi, around 35k
    {"around_price":35000,"exclude_brands":["Xiaomi"],"requirements":["waterproof"],"rank_feature":"battery"}
    matched: ['long lasting', 'rain', 'xiaomi'] 

nothing too fancy, around 20k
    {"around_price":20000}
    matched: [] 

Nothing Phone with a good camera under 40k
    {"max_price":40000,"brands":["Nothing"],"rank_feature":"camera"}
    matched: ['nothing phone', 'camera'] 

waterproof phone with wireless charging and a good camera under 70k
    {"max_price":70000,"requirements":["waterproof","wireless_charging"],"rank_feature":"camera"}
    matched: ['waterproof', 'wireless charging', 'camera'] 



### 8. Paraphrases: the Language Model (Ollama)

Keywords cannot cover every phrasing. "A phone for PUBG" means performance, but contains no feature word. Only in this case, when **no feature word was found**, the language model is asked one narrow question:

> Which one feature does the user care about most, and which exact words in the sentence show it?

The answer is accepted only if:

1. the feature is one of the known features (enforced by the JSON schema), and
2. the quoted words really appear in the sentence and contain at least one meaningful word, not just filler such as "phone", "best", a brand, or a price.

This check stops the model from inventing a feature. For "Samsung phones under 25k", a model might still answer "performance", but it cannot quote any words that support it, so the answer is discarded.

#### Choosing a model

The notebook is designed to run on modest hardware, such as a laptop with an NVIDIA GTX 1650 (4 GB of video memory). The default is **`gemma3:1b`**. Because the model now only answers one small classification question, a 1-billion-parameter model is a reasonable fit.

| Model | Size | Notes |
|---|---|---|
| `gemma3:1b` | 1B parameters | Lightest; default |
| `phi3` | about 3.8B parameters | Better understanding, still small |
| `llama3.1` | 8B parameters | Larger than 4 GB of video memory, so slower on such GPUs |

Set `USE_MODEL = False` to run the notebook with keywords only, without Ollama.

In [80]:
MODEL = "gemma3:1b"
USE_MODEL = True

In [81]:
def installed_models():

    response = ollama.list()
    models = response.models if hasattr(response, "models") else response["models"]

    return [
        getattr(model, "model", None) or model.get("model") or model.get("name")
        for model in models
    ]


if USE_MODEL:
    try:
        available_models = installed_models()
        print("Installed models:", available_models)

        if MODEL not in available_models and f"{MODEL}:latest" not in available_models:
            print(f"\n'{MODEL}' is not installed. Run in a terminal:  ollama pull {MODEL}")
            print("Or set MODEL to one of the installed models above.")

    except Exception as error:
        print("Could not reach Ollama. Start the Ollama app (or run `ollama serve`) and re-run this cell.")
        print("Details:", error)

Installed models: ['gemma3:1b', 'llama3.1:latest', 'phi3:latest', 'llama3:latest']


In [82]:
class FeatureGuess(BaseModel):

    rank_feature: Feature | None = Field(
        None, description="The one feature the user cares about most, or null if none"
    )
    evidence: str = Field(
        "", description="The exact words copied from the request that show this feature"
    )


FEATURE_GUIDE = {
    "battery": "battery life, the phone lasting long",
    "performance": "speed, gaming, heavy apps",
    "camera": "taking photos or recording videos",
    "charging": "charging quickly",
    "brightness": "seeing the screen outdoors or in sunlight",
    "refresh_rate": "smooth scrolling and animations",
    "slimness": "a thin phone",
    "lightness": "a light phone",
    "compactness": "a small phone",
    "zoom": "zooming in on far-away things",
    "multimedia": "watching videos and movies",
    "brand_value": "a prestigious or premium brand",
}

FEATURE_PROMPT = "\n".join(
    [
        "You read one smartphone shopping request and decide which ONE feature matters most to the user.",
        "Choose from these features:",
    ]
    + [f"- {name}: {meaning}" for name, meaning in FEATURE_GUIDE.items()]
    + [
        "",
        "Also copy the exact words from the request that show this feature into 'evidence'.",
        "If the request does not point to any of these features, answer null with empty evidence.",
        "",
        "Examples:",
        'Request: my phone heats up and lags in BGMI',
        '{"rank_feature": "performance", "evidence": "lags in BGMI"}',
        'Request: I want to read maps clearly while cycling at noon',
        '{"rank_feature": "brightness", "evidence": "read maps clearly while cycling at noon"}',
        'Request: just a decent phone for my father',
        '{"rank_feature": null, "evidence": ""}',
    ]
)

print(FEATURE_PROMPT)

You read one smartphone shopping request and decide which ONE feature matters most to the user.
Choose from these features:
- battery: battery life, the phone lasting long
- performance: speed, gaming, heavy apps
- camera: taking photos or recording videos
- charging: charging quickly
- brightness: seeing the screen outdoors or in sunlight
- refresh_rate: smooth scrolling and animations
- slimness: a thin phone
- lightness: a light phone
- compactness: a small phone
- zoom: zooming in on far-away things
- multimedia: watching videos and movies
- brand_value: a prestigious or premium brand

Also copy the exact words from the request that show this feature into 'evidence'.
If the request does not point to any of these features, answer null with empty evidence.

Examples:
Request: my phone heats up and lags in BGMI
{"rank_feature": "performance", "evidence": "lags in BGMI"}
Request: I want to read maps clearly while cycling at noon
{"rank_feature": "brightness", "evidence": "read maps cle

The check on the quoted words uses a list of filler words: brand names, price words, words already used for requirements, processors, or displays, and common words that say nothing about a feature. A quote made only of such words is not evidence for a feature.

In [83]:
FILLER_WORDS = (
    BRAND_WORDS
    | {w for cue in MIN_CUES | MAX_CUES | POST_MAX_CUES | POST_MIN_CUES for w in cue.split()}
    | {w for terms in REQUIREMENT_TERMS.values() for term in terms for w in re.findall(r"[a-z']+", term)}
    | {w for alias in list(PROCESSOR_ALIASES) + list(DISPLAY_ALIASES) for w in re.findall(r"[a-z']+", alias)}
    | {
        "a", "an", "the", "i", "me", "my", "we", "you", "it", "is", "are", "am", "be",
        "need", "want", "looking", "show", "some", "any", "phone", "phones", "mobile",
        "smartphone", "new", "one", "with", "and", "or", "of", "in", "on", "to", "that",
        "good", "best", "great", "nice", "decent", "better", "value", "cheap", "affordable",
        "rs", "inr", "rupees", "k", "lakh", "thousand", "please", "get", "buy", "something"
    }
)


def check_evidence(sentence, evidence):
    """Return None if the quoted words support a feature, otherwise the reason they do not."""

    evidence = evidence.strip().lower()

    if not evidence:
        return "no words were quoted"

    if evidence not in sentence.lower():
        return f'the quoted words "{evidence}" are not in the request'

    meaningful = [
        word for word in re.findall(r"[a-z']+", evidence)
        if word not in FILLER_WORDS
    ]

    if not meaningful:
        return f'the quoted words "{evidence}" are only brand, price, requirement, or filler words'

    return None


def call_model(messages, schema):

    arguments = dict(
        model=MODEL,
        messages=messages,
        format=schema,
        options={"temperature": 0}
    )

    try:
        response = ollama.chat(**arguments, think=False)
    except (TypeError, ollama.ResponseError):
        # Models without a "thinking" mode, or older clients, do not accept the option
        response = ollama.chat(**arguments)

    return response.message.content


def model_guess_feature(sentence):

    answer = call_model(
        [
            {"role": "system", "content": FEATURE_PROMPT},
            {"role": "user", "content": f"Request: {sentence}"}
        ],
        FeatureGuess.model_json_schema()
    )

    try:
        guess = FeatureGuess.model_validate_json(answer)
    except ValidationError:
        return None, "", "the model's answer was not valid"

    if guess.rank_feature is None:
        return None, "", "the model found no feature"

    problem = check_evidence(sentence, guess.evidence)

    if problem:
        return None, guess.evidence, f"the model suggested {guess.rank_feature}, rejected because {problem}"

    return guess.rank_feature, guess.evidence, "accepted"

### 9. Asking in Plain English

`extract_request` combines both steps, and `ask` prints how the sentence was understood before showing the recommendations, so any misreading is visible.

In [84]:
def extract_request(sentence, use_model=None):

    use_model = USE_MODEL if use_model is None else use_model

    request, matched_words = keyword_extract(sentence)
    note = "keywords"

    if request.rank_feature is None and use_model:
        feature, evidence, status = model_guess_feature(sentence)

        if feature:
            request = request.model_copy(update={"rank_feature": feature})
            note = f"language model, based on \"{evidence}\""
        else:
            note = status

    return request, note


def describe_request(request, note):

    lines = [f"budget          : {budget_text(request)}"]

    for field, label in [
        ("brands", "brands"),
        ("exclude_brands", "excluding"),
        ("processor_types", "processor"),
        ("display_types", "display"),
        ("requirements", "requirements"),
    ]:
        values = getattr(request, field)
        if values:
            lines.append(f"{label:<16}: {', '.join(values)}")

    feature = request.rank_feature or "value score only"
    lines.append(f"ranked by       : {feature}  [{note}]")

    if request.other_features:
        lines.append(
            f"also mentioned  : {', '.join(request.other_features)} "
            "(only one feature can be ranked at a time for now)"
        )

    return "\n".join(lines)


def ask(sentence):

    request, note = extract_request(sentence)

    print(f'You asked: "{sentence}"\n')
    print("Understood as:")
    print(describe_request(request, note))
    print()

    return recommend_from_request(request)

#### 9.1 Examples

These sentences contain feature words, so keywords alone handle them and Ollama is not needed:

In [85]:
ask("smallest phone I can get for under 30k")

You asked: "smallest phone I can get for under 30k"

Understood as:
budget          : Rs. 24,000 to Rs. 30,000; up to Rs. 33,000 with possible discounts
ranked by       : compactness  [keywords]

34 phones match the filters (Rs. 24,000 to Rs. 30,000; up to Rs. 33,000 with possible discounts) -> 7 shortlisted for compactness -> 5 recommended


,name,brand,price,size,value_score,note
0,Lava Virat Curve,Lava,24000,6.67,70.3,
1,Oppo F33,Oppo,32000,6.59,40.6,"Could be available at Rs. 30,000 with some dis..."
2,Moto G77 Power,Motorola,26000,6.72,31.6,
3,Moto G Max,Motorola,28000,6.70,23.2,
4,Realme P4x,Realme,25000,6.72,22.6,


In [86]:
ask("I binge watch movies all the time, budget is 40k")

You asked: "I binge watch movies all the time, budget is 40k"

Understood as:
budget          : Rs. 32,000 to Rs. 40,000; up to Rs. 44,000 with possible discounts
ranked by       : multimedia  [keywords]

25 phones match the filters (Rs. 32,000 to Rs. 40,000; up to Rs. 44,000 with possible discounts) -> 5 shortlisted for multimedia -> 5 recommended


,name,brand,price,size,ppi,hbm_filled,value_score,note
0,Vivo T5,Vivo,35000,6.83,450,2000.0,56.1,
1,Iqoo Z11,IQOO,35000,6.83,450,2000.0,54.8,
2,Vivo S2 FE,Vivo,39000,6.83,450,2000.0,27.7,
3,Vivo S2,Vivo,43000,6.89,450,1606.0,14.8,"Could be available at Rs. 40,000 with some dis..."
4,Vivo T5 Pro,Vivo,42000,6.83,450,2000.0,10.3,"Could be available at Rs. 40,000 with some dis..."


In [87]:
ask("a long lasting phone that survives rain and isn't a Xiaomi, around 35k")

You asked: "a long lasting phone that survives rain and isn't a Xiaomi, around 35k"

Understood as:
budget          : Rs. 31,500 to Rs. 38,500
excluding       : Xiaomi
requirements    : waterproof
ranked by       : battery  [keywords]

10 phones match the filters (Rs. 31,500 to Rs. 38,500) -> 5 shortlisted for battery -> 5 recommended


,name,brand,price,battery,value_score
0,Oppo K14 Plus,Oppo,33000,8000,65.8
1,Realme P4s,Realme,35000,8000,51.6
2,Realme P4 Power,Realme,33000,10001,42.6
3,Oneplus Nord CE 6,OnePlus,38000,8000,31.0
4,Realme 16T,Realme,32000,8000,11.0


In [88]:
ask("lightweight amoled phone under 15k")

You asked: "lightweight amoled phone under 15k"

Understood as:
budget          : Rs. 12,000 to Rs. 15,000; up to Rs. 16,500 with possible discounts
display         : Amoled
ranked by       : lightness  [keywords]

No available phone matches all of these requirements.
- Without the budget (Rs. 12,000 to Rs. 15,000; up to Rs. 16,500 with possible discounts), 111 phones would match.
- Without the display choice (Amoled), 13 phones would match.


,name,brand,price,value_score


These contain no feature word, so the language model is asked. For the first two, it should find the feature in the paraphrase. For the rest, the right answer is *no* feature: the request is fully described by brands, requirements, and the budget, and any feature the model suggests should be rejected by the evidence check. The last one is impossible and shows the "no match" explanation.

In [89]:
ask("Need a phone for PUBG under 35 thousand")

You asked: "Need a phone for PUBG under 35 thousand"

Understood as:
budget          : Rs. 28,000 to Rs. 35,000; up to Rs. 38,500 with possible discounts
ranked by       : performance  [language model, based on "PUBG"]

35 phones match the filters (Rs. 28,000 to Rs. 35,000; up to Rs. 38,500 with possible discounts) -> 6 shortlisted for performance -> 5 recommended


,name,brand,price,antutu,value_score,note
0,Moto Edge 70 Fusion,Motorola,31000,1132967,76.1,
1,Nothing Phone 4a,Nothing,37000,1163152,69.0,"Could be available at Rs. 35,000 with some dis..."
2,Vivo T5,Vivo,35000,1111603,56.1,
3,Iqoo Z11,IQOO,35000,1111603,54.8,
4,Realme GT 7T,Realme,35000,1761804,45.8,


In [90]:
ask("I take my phone hiking and need to read the screen at noon, 30k max")

You asked: "I take my phone hiking and need to read the screen at noon, 30k max"

Understood as:
budget          : Rs. 24,000 to Rs. 30,000; up to Rs. 33,000 with possible discounts
ranked by       : value score only  [the model suggested brightness, rejected because the quoted words "reading the screen at noon" are not in the request]

34 phones match the filters (Rs. 24,000 to Rs. 30,000; up to Rs. 33,000 with possible discounts) -> 6 shortlisted for value score -> 5 recommended


,name,brand,price,value_score,note
0,Realme P4 Pro,Realme,29000,98.7,
1,Galaxy F56,Samsung,28000,89.0,
2,Realme Narzo Power,Realme,28000,85.2,
3,Poco X8,POCO,30000,82.6,
4,Redmi Note 15 Pro,Xiaomi,32000,79.4,"Could be available at Rs. 30,000 with some dis..."


In [91]:
ask("my phone fell in the pool again, need a waterproof one under 25k")

You asked: "my phone fell in the pool again, need a waterproof one under 25k"

Understood as:
budget          : Rs. 20,000 to Rs. 25,000; up to Rs. 27,500 with possible discounts
requirements    : waterproof
ranked by       : value score only  [the model suggested brand_value, rejected because the quoted words "waterproof one" are only brand, price, requirement, or filler words]

5 phones match the filters (Rs. 20,000 to Rs. 25,000; up to Rs. 27,500 with possible discounts) -> 5 shortlisted for value score -> 5 recommended


,name,brand,price,value_score
0,Moto Edge 60 Fusion,Motorola,23000,91.6
1,Oppo K14,Oppo,20000,87.1
2,Realme Narzo 100x,Realme,21000,83.9
3,Infinix Hot 70 Pro,Infinix,25000,74.8
4,Realme P4R,Realme,23000,67.7


In [92]:
ask("want something that feels premium and solid, not plasticky, up to 50 thousand")

You asked: "want something that feels premium and solid, not plasticky, up to 50 thousand"

Understood as:
budget          : Rs. 40,000 to Rs. 50,000; up to Rs. 55,000 with possible discounts
requirements    : good_build
ranked by       : value score only  [the model suggested brand_value, rejected because the quoted words "premium and solid" are only brand, price, requirement, or filler words]

8 phones match the filters (Rs. 40,000 to Rs. 50,000; up to Rs. 55,000 with possible discounts) -> 5 shortlisted for value score -> 5 recommended


,name,brand,price,value_score,note
0,Moto Edge 70 Max,Motorola,55000,100.0,"Could be available at Rs. 50,000 with some dis..."
1,Iqoo 15R,IQOO,54000,97.4,"Could be available at Rs. 50,000 with some dis..."
2,Poco X8 Pro Max,POCO,49000,94.8,
3,Oneplus 13s,OnePlus,55000,94.2,"Could be available at Rs. 50,000 with some dis..."
4,Poco X8 Pro,POCO,40000,73.5,


In [93]:
ask("samsung phones under 25k")

You asked: "samsung phones under 25k"

Understood as:
budget          : Rs. 20,000 to Rs. 25,000; up to Rs. 27,500 with possible discounts
brands          : Samsung
ranked by       : value score only  [the model suggested brand_value, rejected because the quoted words "samsung" are only brand, price, requirement, or filler words]

6 phones match the filters (Rs. 20,000 to Rs. 25,000; up to Rs. 27,500 with possible discounts) -> 5 shortlisted for value score -> 5 recommended


,name,brand,price,value_score
0,Galaxy F70,Samsung,20000,92.3
1,Galaxy M17,Samsung,20000,85.8
2,Galaxy F36,Samsung,21000,74.2
3,Galaxy M36,Samsung,21000,66.5
4,Galaxy F17,Samsung,23500,35.5


In [94]:
ask("I need a iphone under 40000")

You asked: "I need a iphone under 40000"

Understood as:
budget          : Rs. 32,000 to Rs. 40,000; up to Rs. 44,000 with possible discounts
brands          : Apple
ranked by       : value score only  [the model suggested brand_value, rejected because the quoted words "iphone" are only brand, price, requirement, or filler words]

No available phone matches all of these requirements.
- Without the budget (Rs. 32,000 to Rs. 40,000; up to Rs. 44,000 with possible discounts), 6 phones would match.
- Without the brand choice (Apple), 31 phones would match.
- Available Apple phones cost Rs. 80,000 to Rs. 180,000.


,name,brand,price,value_score


### 10. Evaluation

Each test sentence is paired with the request a careful human would write. The comparison is **strict**: every field must match, and a field the human left empty must also be empty in the extracted request. (The first, model-only version was scored more leniently, ignoring extra fields, and still reached only 4 of 16.)

There are two test sets:

- **Original set** — the 16 sentences used to test the model-only version. The keyword lists were written while these sentences were known, so this score is optimistic.
- **Held-out set** — 14 sentences written *before* the keyword extractor and not used to tune it. This is the fairer estimate of how well new sentences are understood.

`other_features` is not scored, because which secondary features a person mentions is a matter of interpretation.

In [95]:
TEST_CASES = [
    ("best battery phone under 40k",
     {"max_price": 40000, "rank_feature": "battery"}),
    ("compact phone below 30000",
     {"max_price": 30000, "rank_feature": "compactness"}),
    ("samsung phones under 25k",
     {"max_price": 25000, "brands": ["Samsung"]}),
    ("phones with great brand value under 60k",
     {"max_price": 60000, "rank_feature": "brand_value"}),
    ("good for watching youtube and netflix, under 35k",
     {"max_price": 35000, "rank_feature": "multimedia"}),
    ("phone with good sound for music under 20000",
     {"max_price": 20000, "requirements": ["stereo_speakers"]}),
    ("waterproof phone within 25k",
     {"max_price": 25000, "requirements": ["waterproof"]}),
    ("solid premium build, not plastic, max 50k",
     {"max_price": 50000, "requirements": ["good_build"]}),
    ("anything except vivo and oppo with fast charging below 30k",
     {"max_price": 30000, "exclude_brands": ["Oppo", "Vivo"], "rank_feature": "charging"}),
    ("redmi phone with big battery under 20k",
     {"max_price": 20000, "brands": ["Xiaomi"], "rank_feature": "battery"}),
    ("iphone around 1 lakh",
     {"around_price": 100000, "brands": ["Apple"]}),
    ("gaming phone between 30k and 50k",
     {"min_price": 30000, "max_price": 50000, "rank_feature": "performance"}),
    ("thinnest phone above 20000",
     {"min_price": 20000, "rank_feature": "slimness"}),
    ("pixel with the best zoom",
     {"brands": ["Google"], "rank_feature": "zoom"}),
    ("lightweight amoled phone under 15k",
     {"max_price": 15000, "display_types": ["Amoled"], "rank_feature": "lightness"}),
    ("waterproof phone with wireless charging and a good camera under 70k",
     {"max_price": 70000, "requirements": ["waterproof", "wireless_charging"], "rank_feature": "camera"}),
]

HELD_OUT_CASES = [
    ("Need a phone for PUBG under 35 thousand",
     {"max_price": 35000, "rank_feature": "performance"}),
    ("Which oneplus has the biggest battery below Rs 45,000?",
     {"max_price": 45000, "brands": ["OnePlus"], "rank_feature": "battery"}),
    ("looking for a slim phone, not motorola, budget 25000",
     {"max_price": 25000, "exclude_brands": ["Motorola"], "rank_feature": "slimness"}),
    ("best camera phone from vivo or oppo, 50k max",
     {"max_price": 50000, "brands": ["Oppo", "Vivo"], "rank_feature": "camera"}),
    ("something with stereo speakers and an AMOLED display around 20k",
     {"around_price": 20000, "requirements": ["stereo_speakers"], "display_types": ["Amoled"]}),
    ("I want a water resistant phone with a big screen for movies under 30k",
     {"max_price": 30000, "requirements": ["waterproof"], "rank_feature": "multimedia"}),
    ("fastest charging phone between 15k and 25k",
     {"min_price": 15000, "max_price": 25000, "rank_feature": "charging"}),
    ("a light phone that fits in my pocket, under 20000",
     {"max_price": 20000, "rank_feature": "lightness"}),
    ("phone with a good telephoto lens above 60k",
     {"min_price": 60000, "rank_feature": "zoom"}),
    ("mediatek gaming phone under 25k",
     {"max_price": 25000, "processor_types": ["Mediatek"], "rank_feature": "performance"}),
    ("show me premium brand phones under 80000",
     {"max_price": 80000, "rank_feature": "brand_value"}),
    ("sturdy phone with wireless charging, 1 lakh budget",
     {"max_price": 100000, "requirements": ["good_build", "wireless_charging"]}),
    ("my phone dies before lunch, need a new one around 18k",
     {"around_price": 18000, "rank_feature": "battery"}),
    ("pixel or galaxy with a bright screen under 70k",
     {"max_price": 70000, "brands": ["Google", "Samsung"], "rank_feature": "brightness"}),
]

SCORED_FIELDS = [
    "max_price", "min_price", "around_price", "brands", "exclude_brands",
    "processor_types", "display_types", "requirements", "rank_feature"
]


def strict_compare(request, expected):

    wrong = []

    for field in SCORED_FIELDS:
        actual = getattr(request, field)
        wanted = expected.get(field, [] if isinstance(actual, list) else None)

        same = set(actual) == set(wanted) if isinstance(actual, list) else actual == wanted

        if not same:
            wrong.append(field)

    return wrong


def evaluate(cases, use_model):

    rows = []

    for sentence, expected in cases:
        request, note = extract_request(sentence, use_model=use_model)
        wrong = strict_compare(request, expected)

        rows.append({
            "sentence": sentence,
            "correct": not wrong,
            "wrong fields": ", ".join(wrong),
            "feature from": note,
            "extracted": request.model_dump_json(exclude_defaults=True, exclude={"other_features"})
        })

    return pd.DataFrame(rows)

#### 10.1 Keywords only (no language model)

In [96]:
for name, cases in [("Original set", TEST_CASES), ("Held-out set", HELD_OUT_CASES)]:
    result = evaluate(cases, use_model=False)
    print(f"{name}: {result['correct'].sum()} / {len(result)} fully correct")

Original set: 16 / 16 fully correct
Held-out set: 13 / 14 fully correct


In [97]:
held_out_keywords = evaluate(HELD_OUT_CASES, use_model=False)

held_out_keywords[["sentence", "correct", "wrong fields", "extracted"]]

,sentence,correct,wrong fields,extracted
0,Need a phone for PUBG under 35 thousand,False,rank_feature,"{""max_price"":35000}"
1,Which oneplus has the biggest battery below Rs...,True,,"{""max_price"":45000,""brands"":[""OnePlus""],""rank_..."
2,"looking for a slim phone, not motorola, budget...",True,,"{""max_price"":25000,""exclude_brands"":[""Motorola..."
3,"best camera phone from vivo or oppo, 50k max",True,,"{""max_price"":50000,""brands"":[""Vivo"",""Oppo""],""r..."
4,something with stereo speakers and an AMOLED d...,True,,"{""around_price"":20000,""display_types"":[""Amoled..."
5,I want a water resistant phone with a big scre...,True,,"{""max_price"":30000,""requirements"":[""waterproof..."
6,fastest charging phone between 15k and 25k,True,,"{""max_price"":25000,""min_price"":15000,""rank_fea..."
7,"a light phone that fits in my pocket, under 20000",True,,"{""max_price"":20000,""rank_feature"":""lightness""}"
8,phone with a good telephoto lens above 60k,True,,"{""min_price"":60000,""rank_feature"":""zoom""}"
9,mediatek gaming phone under 25k,True,,"{""max_price"":25000,""processor_types"":[""Mediate..."


#### 10.2 Keywords + language model

The model is only consulted for sentences without a feature word, so differences from 10.1 come from those sentences. A good model fixes paraphrases without breaking sentences that should have no feature.

In [98]:
print(f"Model: {MODEL}\n")

for name, cases in [("Original set", TEST_CASES), ("Held-out set", HELD_OUT_CASES)]:
    result = evaluate(cases, use_model=True)
    print(f"{name}: {result['correct'].sum()} / {len(result)} fully correct")

    consulted = result[~result["feature from"].eq("keywords")]
    if not consulted.empty:
        display(consulted[["sentence", "correct", "feature from"]])

Model: gemma3:1b



Original set: 16 / 16 fully correct


,sentence,correct,feature from
2,samsung phones under 25k,True,"the model suggested brand_value, rejected beca..."
5,phone with good sound for music under 20000,True,"the model suggested multimedia, rejected becau..."
6,waterproof phone within 25k,True,"the model suggested battery, rejected because ..."
7,"solid premium build, not plastic, max 50k",True,"the model suggested brand_value, rejected beca..."
10,iphone around 1 lakh,True,"the model suggested brand_value, rejected beca..."


Held-out set: 14 / 14 fully correct


,sentence,correct,feature from
0,Need a phone for PUBG under 35 thousand,True,"language model, based on ""PUBG"""
4,something with stereo speakers and an AMOLED d...,True,"the model suggested multimedia, rejected becau..."
11,"sturdy phone with wireless charging, 1 lakh bu...",True,"the model suggested brand_value, rejected beca..."


### 11. Limitations

- **Keyword lists are hand-written.** They cover common phrasings, and the held-out score estimates how often new sentences are fully understood. Unusual wording can still be missed; such sentences are worth adding to the lists and to the test sets.
- **The language model only fills in the main feature.** It cannot add brands, prices, or requirements that the keywords missed.
- **One ranking feature at a time.** Further features are reported as "also mentioned" but not used yet.
- **The discount note is an assumption.** Phones up to 10% above the budget are shown with a note that they *could* be available within budget through discounts; the dataset holds no actual offer data.
- **"Under X" searches 0.8X to X.** A clearly better phone just below 0.8X is not considered; asking "between Rs. A and Rs. B" widens the range.
- **Brand value is the brand's mean price** among available phones. Brands with few phones (for example Infinix or Lava) have a less stable estimate.
- **Screen size dominates the multimedia rule.** A large, lower-resolution phone can rank above a slightly smaller, sharper one.
- **Filled brightness values are estimates**, based on the display-type mean.
- **"Good build" depends on recorded fields.** A phone with an empty `protection` value is not considered well built.
- **Prices are a snapshot** of the values recorded in the dataset.

### 12. Streamlit App

The same pipeline is available as a small web app in the `app/` folder:

```text
app/
├── smartphone_assistant.py   # the logic of this notebook: data, keywords, model fallback, rules
└── app.py                    # the Streamlit interface
```

`smartphone_assistant.py` contains the same rules, keyword lists, and model check as Sections 2–9, so the app and this notebook give the same answers.

To run it, install Streamlit once and start the app from the project root:

```bash
pip install streamlit
streamlit run app/app.py
```

The app opens in the browser. Type a request, and it shows how the request was understood, the recommended phones, or the reason nothing matches. The sidebar sets the Ollama model and can switch the model off, in which case only keywords are used.